# 📁 线性神经网络：整章实践探索项目

本笔记本覆盖**第 3 章「线性神经网络」全部内容**——从线性回归到 softmax 分类。
设计了 **3 个项目**，每个项目的目标不是「把习题做对」，而是**像一个研究者一样去探索、观察和思考**。

> | 项目 | 主题 | 覆盖章节 |
> |------|------|---------|
> | 🕵️ 项目 1 | 数据侦探：解密未知数据集 | 3.1—3.4 |
> | 💥 项目 2 | 数值灾难与救援：softmax 的软肋 | 3.4, 3.6—3.7 |
> | 🎨 项目 3 | 线性模型的边界：Fashion-MNIST 大探索 | 3.4—3.7 |

每个项目预计用时 30—60 分钟。项目 1 和 2 相对独立，可按任意顺序完成；项目 3 建议最后做。

In [1]:
%matplotlib inline
import math, random, time
import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils import data
from torchvision import transforms
from d2l import torch as d2l
import matplotlib.pyplot as plt

---

## 🕵️ 项目 1：数据侦探——解密未知数据集

### 🎯 场景

你收到了一份匿名数据集 `X` 和 `y`，没有附带任何说明。
你的任务是：**仅通过数据分析和建模，推断出这份数据的一切——它是回归问题还是分类问题？
如果是回归，噪声有多大？如果是分类，有几类？线性模型能解决到什么程度？**

下面已经为你准备好了两份「密档」。请逐一破解。

### 🔐 密档生成器（不要偷看！）

运行下面的代码来生成你的密档。每个密档都由随机参数生成，所以每次运行都会得到不同的数据。

In [2]:
# ===== 运行此 cell 生成密档，不要修改！=====
torch.manual_seed(42)  # 固定种子使结果可复现

def make_mystery_1():
    """密档 #1：回归还是分类？"""
    true_w = torch.tensor([1.5, -2.7, 0.8])
    true_b = 0.5
    X = torch.normal(0, 1, (500, 3))
    y = torch.matmul(X, true_w) + true_b
    y += torch.normal(0, 0.3, y.shape)  # 加了一些噪声
    return X, y.reshape((-1, 1))

def make_mystery_2():
    """密档 #2：分类问题——究竟有几类？"""
    num_classes = 4
    num_features = 2
    # 每个类有不同的均值，使它们在空间中分开
    centers = torch.tensor([[2.0, 2.0], [-2.0, 2.0], [2.0, -2.0], [-2.0, -2.0]])
    X_list, y_list = [], []
    for i in range(num_classes):
        n = 200
        X_list.append(centers[i] + torch.normal(0, 0.8, (n, 2)))
        y_list.append(torch.full((n,), i, dtype=torch.long))
    X = torch.cat(X_list, dim=0)
    y = torch.cat(y_list, dim=0)
    # 打乱
    idx = torch.randperm(len(y))
    return X[idx], y[idx]

X1, y1 = make_mystery_1()
X2, y2 = make_mystery_2()

print(f'密档 #1: X shape = {X1.shape}, y shape = {y1.shape}')
print(f'  y 的范围: [{y1.min().item():.2f}, {y1.max().item():.2f}]')
print(f'  y 的唯一值数量: {len(torch.unique(y1))}')
print()
print(f'密档 #2: X shape = {X2.shape}, y shape = {y2.shape}')
print(f'  y 的唯一值: {torch.unique(y2).tolist()}')
print(f'  y 的 dtype: {y2.dtype}')

密档 #1: X shape = torch.Size([500, 3]), y shape = torch.Size([500, 1])
  y 的范围: [-9.62, 9.78]
  y 的唯一值数量: 500

密档 #2: X shape = torch.Size([800, 2]), y shape = torch.Size([800])
  y 的唯一值: [0, 1, 2, 3]
  y 的 dtype: torch.int64


### 📝 任务 1.1：判断密档 #1 的问题类型

只看 `y1` 的值——它是连续值还是离散值？这个问题应该用回归还是分类？为什么？

然后用线性回归拟合它，观察：
1. 训练 loss 能降到多低？
2. 残差（预测值 - 真实值）的分布是什么样的？它是否接近正态分布？
3. 残差的标准差能告诉我们什么？（提示：和生成数据时加的噪声有关）

In [ ]:
# === 你的分析代码 ===

# 1. 先用线性回归（从零实现或框架都行）拟合 X1, y1
# 2. 观察残差分布
# 3. 估计噪声标准差

# 加载数据
def load_array(data_arrays, batch_size, is_train=True):
    '''构造一个PyTorch数据迭代器'''
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

train_num = 400 + 1
train_X1 = X1[:train_num]
train_y1 = y1[:train_num]
test_X1 = X1[train_num:]
test_y1 = y1[train_num:]
train_iter = load_array((train_X1, train_y1), 40)
# test_iter = load_array((test_X1, test_y1), 40, is_train=False)

# 创建并初始化神经网络
net = nn.Sequential(nn.Linear(3, 1))

net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.zero_()

# 定义损失函数
loss = nn.MSELoss()

# 定义优化算法
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 执行训练
num_epochs = 10
for epoch in range(num_epochs):
    for X, y in train_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    # 在整个测试集上计算 loss（避免只取一个 batch）
    with torch.no_grad():
        test_l = loss(net(test_X1), test_y1)
    print(f'epoch {epoch + 1}, loss {test_l:.6f}')
# 残差分析（修复了广播 bug）
with torch.no_grad():
    # 训练集残差 —— reshape 保证形状一致
    train_pred = net(train_X1).reshape(-1)
    train_y1_flat = train_y1.reshape(-1)
    train_res = train_y1_flat - train_pred
    # 测试集残差
    test_pred = net(test_X1).reshape(-1)
    test_y1_flat = test_y1.reshape(-1)
    test_res = test_y1_flat - test_pred

    print("\n=== 训练集 ===")
    print(f"残差均值: {train_res.mean().item():.6f}")
    print(f"残差标准差: {train_res.std().item():.6f}")
    print(f"RMSE: {torch.sqrt(torch.mean(train_res ** 2)).item():.6f}")
    print(f"R²: {1 - (train_res**2).sum() / ((train_y1_flat - train_y1_flat.mean())**2).sum():.6f}")

    print("\n=== 测试集 ===")
    print(f"残差均值: {test_res.mean().item():.6f}")
    print(f"残差标准差: {test_res.std().item():.6f}")
    print(f"RMSE: {torch.sqrt(torch.mean(test_res ** 2)).item():.6f}")
    print(f"R²: {1 - (test_res**2).sum() / ((test_y1_flat - test_y1_flat.mean())**2).sum():.6f}")

epoch 1, loss 3.023428
epoch 2, loss 0.862277
epoch 3, loss 0.299740
epoch 4, loss 0.185009
epoch 5, loss 0.125263
epoch 6, loss 0.103225
epoch 7, loss 0.095512
epoch 8, loss 0.092872
epoch 9, loss 0.091893
epoch 10, loss 0.091580

=== 训练集 ===
残差均值: 0.015792
残差标准差: 0.301380
RMSE: 0.301418
R²: 0.990938

=== 测试集 ===
残差均值: -0.013386
残差标准差: 0.303865
RMSE: 0.302622
R²: 0.991134


### 📝 任务 1.2：破解密档 #2

`y2` 的 dtype 是整数，唯一值有 4 个——明显是分类问题。但你还需要回答：

1. **可视化**：画出 `X2` 的散点图，用颜色区分 4 个类别。类别之间是线性可分的吗？
2. **训练 softmax 分类器**（从零实现），观察训练精度和测试精度的变化。
3. **找出决策边界**：在二维平面上画出模型学到的 4 个类别的决策区域。
   提示：在平面上取密集的网格点，用模型预测每个点的类别，然后用 `plt.contourf` 画出。
4. **如果有些类别总是被分错**，你能从散点图中看出原因吗？

In [ ]:
# === 你的分析代码 ===

# 1. 可视化散点图
# 2. 划分训练/测试集
# 3. 训练 softmax 分类器（从零实现）
# 4. 画出决策边界

# TODO
pass

### 📝 任务 1.3（挑战）：用错误模型会发生什么？

这是一个思想实验 + 代码验证：

1. **在分类数据（密档 #2）上跑线性回归**（把类别标签当作连续值）。
   训练出来的模型给出的预测值可能不是 0/1/2/3 中的任何一个——你该如何把它变成分类结果？
2. **在回归数据（密档 #1）上跑 softmax 分类**。
   你需要先把连续的 `y1` 离散化（分箱），然后训练分类器。这样做损失了什么信息？

> 💡 这个实验让你理解**选错模型**的代价。在实际项目中，识别问题类型是第一步。

In [ ]:
# === 你的分析代码 ===

# 1. 在分类数据上跑线性回归
# 2. 在回归数据上跑 softmax（先分箱）

# TODO
pass

### 🤔 侦探笔记

1. 在密档 #1 中，你观察到的残差标准差大概是多少？如果残差的标准差远大于或远小于你预估的噪声水平，分别意味着什么？
2. 密档 #2 中，如果两个类的中心靠得很近（比如都在原点附近），softmax 分类器还能有效区分它们吗？你如何量化分类的难度？
3. 如果有人说「我把所有问题都当回归来做」，你会怎么反驳他？至少给出两个理由。

---

## 💥 项目 2：数值灾难与救援——当 softmax 爆炸时

### 🎯 场景

教材 3.7 节提到：直接计算 $\frac{\exp(o_j)}{\sum_k \exp(o_k)}$ 可能导致**数值上溢**（overflow）
和**下溢**（underflow）。但这听起来很抽象——到底什么时候会炸？炸出来是什么样的？

在本项目中，你将自己制造这场「数值灾难」，亲眼看到 `inf` 和 `nan` 的出现，
然后亲手实现 LogSumExp 技巧来拯救它。

### 📝 任务 2.1：制造灾难——找到 softmax 的「爆炸点」

编写一个**朴素的 softmax 函数**（不做任何数值保护），然后测试：

1. 当输入为 `[0, 0, 0]` 时，输出是什么？
2. 当输入为 `[10, 20, 30]` 时呢？
3. 当输入为 `[100, 200, 300]` 时呢？
4. 不断增大输入值（每次增加 10 倍），**找到 softmax 开始返回 NaN 的临界值**。

同样地，测试非常负的输入（如 `[-500, -600, -700]`）——什么时候开始出现全是 0 的输出？

In [ ]:
# === 你的实验代码 ===

def naive_softmax(x):
    """朴素实现——没有任何数值保护"""
    # TODO
    pass

# 测试不同量级的输入
test_inputs = [
    torch.tensor([0., 0., 0.]),
    torch.tensor([10., 20., 30.]),
    torch.tensor([100., 200., 300.]),
    # ... 继续增大
]

for x in test_inputs:
    y = naive_softmax(x)
    print(f'输入: {x.tolist()}')
    print(f'输出: {y.tolist()}, 和: {y.sum().item():.4f}')
    print(f'有 NaN? {torch.isnan(y).any().item()}, 有 Inf? {torch.isinf(y).any().item()}')
    print()

### 📝 任务 2.2：可视化「安全区」和「危险区」

考虑一维 softmax（即 sigmoid）：$\sigma(x) = \frac{e^x}{1 + e^x}$。

在 $x \in [-100, 100]$ 范围内，分别用**朴素版本**和**稳定版本**
（先减最大值）计算 $\sigma(x)$，画出两条曲线对比。

然后扩展范围到 $x \in [-1000, 1000]$——观察朴素版本在哪里开始出错。

In [ ]:
# === 你的可视化代码 ===

def naive_sigmoid(x):
    """朴素 sigmoid"""
    return 1 / (1 + torch.exp(-x))

def stable_sigmoid(x):
    """数值稳定的 sigmoid"""
    # TODO: 实现稳定版本
    # 提示：对 x > 0 和 x <= 0 分别处理
    pass

# TODO: 在 [-100, 100] 和 [-1000, 1000] 两个范围上画图对比
pass

### 📝 任务 2.3：实现 LogSumExp 并验证梯度

教材 3.7 节介绍了把 softmax 和交叉熵合并计算可以避免数值问题。
其核心是 **LogSumExp** 技巧：

$$\text{LogSumExp}(\mathbf{o}) = \max_k o_k + \log\left(\sum_k \exp(o_k - \max_j o_j)\right)$$

请实现：
1. **朴素的交叉熵**：先 `softmax`，再 `-log` 取正确类别的概率
2. **稳定的交叉熵**：直接用 LogSumExp 计算
3. 在大数值输入下（如 $o = [500, 600, 700]$），对比两者的输出和梯度

In [ ]:
# === 你的实现代码 ===

def logsumexp(x, dim=-1):
    """数值稳定的 LogSumExp"""
    # TODO
    pass

def naive_cross_entropy(logits, y):
    """朴素交叉熵：softmax → -log(正确类别概率)"""
    # TODO
    pass

def stable_cross_entropy(logits, y):
    """稳定交叉熵：使用 LogSumExp 技巧"""
    # TODO
    pass

# 测试
# 小数值——两者应该一致
logits_small = torch.tensor([[1.0, 2.0, 3.0], [4.0, 1.0, 2.0]])
labels = torch.tensor([2, 0])
print('小数值测试:')
print(f'  朴素版: {naive_cross_entropy(logits_small, labels)}')
print(f'  稳定版: {stable_cross_entropy(logits_small, labels)}')

# 大数值——朴素版应该爆炸
logits_large = torch.tensor([[500., 600., 700.], [700., 500., 600.]])
labels = torch.tensor([2, 0])
print('\n大数值测试:')
print(f'  朴素版: {naive_cross_entropy(logits_large, labels)}')
print(f'  稳定版: {stable_cross_entropy(logits_large, labels)}')

### 📝 任务 2.4：与 PyTorch 内置实现对比

PyTorch 的 `nn.CrossEntropyLoss` 内部做了数值保护。请：

1. 用 `nn.CrossEntropyLoss(reduction='none')` 计算上面的 loss，与你的稳定版本对比
2. 阅读 [PyTorch 文档](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 中关于数值稳定性的说明
3. 思考：为什么教材 3.6 节从零实现时没有遇到数值问题，但在 3.7 节特别强调了这个问题？

In [ ]:
# === 你的验证代码 ===

# TODO: 对比 PyTorch 内置 CrossEntropyLoss
pass

### 🤔 灾难报告

1. 在你的实验中，softmax 大约在输入值超过多少时开始出现 NaN？这个阈值和浮点数表示范围有什么关系？（提示：`torch.finfo(torch.float32).max`）
2. 如果你在训练一个真实的深度神经网络，loss 突然变成 NaN，你应该检查哪些地方？数值不稳定可能是罪魁祸首吗？
3. 为什么教材 3.6 节（从零实现）没有遇到这个问题，但 3.7 节（简洁实现）却特别强调？在 3.6 节的代码中，什么样的输入可能触发这个问题？

---

## 🎨 项目 3：线性模型的边界——Fashion-MNIST 深度探索

### 🎯 场景

教材在 Fashion-MNIST 上用 softmax 回归达到了约 85% 的测试精度。
但这 85% 是怎么来的？哪 15% 分错了？为什么分错？
线性模型的能力边界到底在哪里？

本项目将带你**深入模型的「内心」**，用混淆矩阵、特征消融等手段，
理解线性分类器在真实数据上的行为。

### 准备工作：加载数据并训练一个基准模型

先快速搭建一个 softmax 回归模型（从零实现或框架皆可），在 Fashion-MNIST 上训练 10 个 epoch，
达到约 83—86% 的测试精度。这个模型将作为你后续分析的「研究对象」。

> 💡 建议使用框架实现（3.7 节），这样可以集中精力在分析上。

In [ ]:
# === 训练基准模型 ===

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

# TODO: 搭建并训练 softmax 回归模型
# 你可以使用 nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
# 配合 nn.CrossEntropyLoss() 和 torch.optim.SGD

pass

### 📝 任务 3.1：混淆矩阵——哪些类别最容易搞混？

计算模型在**测试集**上的混淆矩阵（10×10）。

1. 用 `plt.imshow` 可视化混淆矩阵（用热力图，标注每个格子的数值）
2. 找出**最容易混淆的 3 对类别**（对角线以外的最大值）
3. 从数据集中找出几个被分错的样本，**打印出图片**。你作为人类，能正确分类它们吗？

> 💡 Fashion-MNIST 的 10 个类别：t-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot

In [ ]:
# === 你的混淆矩阵分析 ===

class_names = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
               'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']

# TODO: 1. 计算混淆矩阵
# TODO: 2. 可视化
# TODO: 3. 展示被分错的样本图片

pass

### 📝 任务 3.2：像素消融实验——模型到底在看哪些像素？

线性模型有一个独特优势：**每个像素有一个权重**，直观可解释。

1. **可视化每个类别的权重**：将 784 维的权重向量 reshape 成 28×28，
   对 10 个类别分别画出来（像一张灰度图）。亮色 = 正权重（支持这个类），
   暗色 = 负权重（反对这个类）。你能否从这些「权重图」中看出每个类别的「模板」？
   
2. **像素重要性排序**：对某个类别（如 shirt），找出权重绝对值最大的 20 个像素和最小的 20 个像素，
   在原图上高亮它们。这些像素在图像的什么位置？符合你的直觉吗？

3. **消融实验**：如果只保留权重绝对值最大的 $k$ 个像素（把其余像素设为 0），
   模型精度如何随 $k$ 变化？尝试 $k \in \{10, 20, 50, 100, 200, 400, 784\}$，
   画出精度 vs $k$ 曲线。多少个像素就够 80% 精度了？

In [ ]:
# === 权重可视化与消融实验 ===

# TODO: 1. 可视化 10 个类别各自的 28×28 权重图
# TODO: 2. 找出并高亮最重要的像素
# TODO: 3. 像素消融：精度 vs 保留像素数 k

pass

### 📝 任务 3.3（挑战）：如果标签是 one-hot 向量——回归能替代分类吗？

一个经常被问到的问题：**如果把分类标签变成 one-hot 向量，然后用线性回归（MSE loss）训练，
效果会怎样？**

请你动手验证：
1. 保持模型结构不变（784 输入 → 10 输出），但把损失函数从 CrossEntropyLoss 换成 MSELoss
2. 训练相同 epoch 数，比较两种损失函数的测试精度和学习曲线
3. MSE 训练出来的模型，softmax 之后也是概率分布吗？它的输出有什么不同？
4. 解释你观察到的差异——为什么交叉熵在分类问题上优于 MSE？

In [ ]:
# === MSE vs CrossEntropy 对比实验 ===

# TODO: 用 MSELoss 训练同样的模型，对比精度和收敛速度

pass

### 🤔 反思报告

1. 从混淆矩阵来看，shirt（衬衫）和 t-shirt（T恤）经常被搞混。作为人类，你能从 28×28 的低分辨率图片中区分它们吗？线性模型是不是「情有可原」？
2. 从权重图来看，每个类别学到的「模板」大致是什么样子？为什么鞋类的模板和上衣类的模板看起来完全不同？
3. 像素消融实验的结果说明了什么？线性模型是在「理解」图像，还是在做「模板匹配」？
4. 本章的线性模型是全书最简单的模型。回顾你在这 3 个项目中做的实验，你认为了解线性模型的这些细节，对你学习后续更复杂的模型有什么帮助？

---

## 📖 后记

这三个项目覆盖了第 3 章的核心主题：

| 你学到的东西 | 来自 |
|------------|------|
| 如何区分回归问题和分类问题 | 项目 1 |
| 用残差分析推断噪声特性 | 项目 1 |
| 可视化分类决策边界 | 项目 1 |
| 浮点数精度与数值稳定性 | 项目 2 |
| LogSumExp 技巧的原理和实现 | 项目 2 |
| 混淆矩阵与模型诊断 | 项目 3 |
| 线性模型权重的可解释性 | 项目 3 |
| MSE vs 交叉熵的实际对比 | 项目 3 |

这些不仅是「练习题」——它们是深度学习实践中每天都会遇到的真实问题。
当你进入第 4 章（多层感知机）之后，你会发现：**同样的方法论，
只是模型变得更复杂了。**

> 🔗 参考章节：
> - [3.1 线性回归](https://zh.d2l.ai/chapter_linear-networks/linear-regression.html)
> - [3.2 线性回归的从零开始实现](https://zh.d2l.ai/chapter_linear-networks/linear-regression-scratch.html)
> - [3.3 线性回归的简洁实现](https://zh.d2l.ai/chapter_linear-networks/linear-regression-concise.html)
> - [3.4 softmax 回归](https://zh.d2l.ai/chapter_linear-networks/softmax-regression.html)
> - [3.5 图像分类数据集](https://zh.d2l.ai/chapter_linear-networks/image-classification-dataset.html)
> - [3.6 softmax 回归的从零开始实现](https://zh.d2l.ai/chapter_linear-networks/softmax-regression-scratch.html)
> - [3.7 softmax 回归的简洁实现](https://zh.d2l.ai/chapter_linear-networks/softmax-regression-concise.html)